# Assignment 2: Data Engineering with PySpark

## Overview
This assignment focuses on building a data engineering pipeline using PySpark to process and analyze user activity data.

## Objectives
- Set up PySpark environment
- Load and explore data
- Perform data transformations
- Analyze user behavior patterns
- Export processed data

## Part 1: Environment Setup

In [1]:
# Install required packages
!pip install pyspark pandas numpy matplotlib seaborn

In [2]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Assignment2_DataEngineering") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark session created successfully")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/28 19:16:35 WARN Utils: Your hostname, sable-ThinkPad-X1-Yoga-3rd, resolves to a loopback address: 127.0.1.1; using 10.192.33.105 instead (on interface wlp2s0)
25/12/28 19:16:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/28 19:16:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session created successfully
Spark version: 4.0.1


## Part 2: Data Loading and Exploration

In [4]:
# Define schema for user activity data
schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("username", StringType(), False),
    StructField("action", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("device", StringType(), True),
    StructField("location", StringType(), True),
    StructField("duration_seconds", IntegerType(), True)
])

print("Schema defined")

Schema defined


In [5]:
# Load data from CSV file
print("Loading data...")
df = spark.read.csv(
    "data/user_activity.csv",
    header=True,
    schema=schema
)

print("Data loaded successfully")
print(f"Total records: {df.count()}")

Loading data...


25/12/28 19:16:40 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/user_activity.csv.
java.io.FileNotFoundException: File data/user_activity.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Resolve

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/sable/Documents/data engineering1/lab2-practice/data/user_activity.csv. SQLSTATE: 42K03

In [ ]:
# Display first few records
print("Sample data:")
df.show(10, truncate=False)

In [ ]:
# Display schema
print("Data schema:")
df.printSchema()

In [ ]:
# Basic statistics
print("Basic statistics:")
df.describe().show()

## Part 3: Data Quality Checks

In [ ]:
# Check for null values
print("Checking for null values...")
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

In [ ]:
# Check for duplicate records
print("Checking for duplicates...")
total_records = df.count()
distinct_records = df.distinct().count()
duplicate_count = total_records - distinct_records

print(f"Total records: {total_records}")
print(f"Distinct records: {distinct_records}")
print(f"Duplicate records: {duplicate_count}")

if duplicate_count == 0:
    print("No duplicates found")
else:
    print(f"Found {duplicate_count} duplicate records")

## Part 4: Data Transformations

In [ ]:
# Extract date and time components
print("Extracting date/time components...")
df_transformed = df.withColumn("date", to_date(col("timestamp"))) \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("timestamp"))) \
    .withColumn("month", month(col("timestamp")))

print("Date/time components extracted")
df_transformed.show(5)

In [ ]:
# Categorize duration into time buckets
print("Categorizing durations...")
df_transformed = df_transformed.withColumn(
    "duration_category",
    when(col("duration_seconds") < 60, "short")
    .when((col("duration_seconds") >= 60) & (col("duration_seconds") < 300), "medium")
    .otherwise("long")
)

print("Duration categories created")
df_transformed.groupBy("duration_category").count().show()

## Part 5: Data Analysis

In [ ]:
# User activity analysis
print("Analyzing user activity...")
user_stats = df_transformed.groupBy("user_id", "username").agg(
    count("*").alias("total_actions"),
    sum("duration_seconds").alias("total_duration"),
    avg("duration_seconds").alias("avg_duration"),
    countDistinct("action").alias("unique_actions"),
    countDistinct("device").alias("unique_devices")
).orderBy(col("total_actions").desc())

print("Top 10 most active users:")
user_stats.show(10)

In [ ]:
# Action type analysis
print("Analyzing action types...")
action_stats = df_transformed.groupBy("action").agg(
    count("*").alias("count"),
    avg("duration_seconds").alias("avg_duration")
).orderBy(col("count").desc())

print("Action statistics:")
action_stats.show()

In [ ]:
# Device usage analysis
print("Analyzing device usage...")
device_stats = df_transformed.groupBy("device").agg(
    count("*").alias("usage_count"),
    countDistinct("user_id").alias("unique_users")
).orderBy(col("usage_count").desc())

print("Device statistics:")
device_stats.show()

In [ ]:
# Location analysis
print("Analyzing locations...")
location_stats = df_transformed.groupBy("location").agg(
    count("*").alias("activity_count"),
    countDistinct("user_id").alias("unique_users")
).orderBy(col("activity_count").desc())

print("Top locations:")
location_stats.show(10)

In [ ]:
# Hourly activity pattern
print("Analyzing hourly patterns...")
hourly_stats = df_transformed.groupBy("hour").agg(
    count("*").alias("activity_count")
).orderBy("hour")

print("Hourly activity distribution:")
hourly_stats.show(24)

## Part 6: Advanced Analytics

In [ ]:
# User engagement score calculation
print("Calculating user engagement scores...")
engagement_df = df_transformed.groupBy("user_id", "username").agg(
    count("*").alias("activity_count"),
    sum("duration_seconds").alias("total_time"),
    countDistinct("date").alias("active_days")
)

# Calculate engagement score (normalized)
engagement_df = engagement_df.withColumn(
    "engagement_score",
    (col("activity_count") * 0.4 + col("total_time") / 60 * 0.3 + col("active_days") * 10 * 0.3)
).orderBy(col("engagement_score").desc())

print("Top engaged users:")
engagement_df.show(10)

In [ ]:
# Peak activity times
print("Identifying peak activity times...")
peak_times = df_transformed.groupBy("hour", "day_of_week").agg(
    count("*").alias("activity_count")
).orderBy(col("activity_count").desc())

print("Peak activity times:")
peak_times.show(10)

## Part 7: Data Visualization

In [ ]:
# Convert to Pandas for visualization
print("Preparing data for visualization...")
hourly_pd = hourly_stats.toPandas()
action_pd = action_stats.toPandas()
device_pd = device_stats.toPandas()

print("Data converted to Pandas")

In [ ]:
# Hourly activity plot
plt.figure(figsize=(12, 6))
plt.plot(hourly_pd['hour'], hourly_pd['activity_count'], marker='o', linewidth=2)
plt.title('User Activity by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Activity Count')
plt.grid(True, alpha=0.3)
plt.xticks(range(24))
plt.tight_layout()
plt.show()

print("Hourly activity plot created")

In [ ]:
# Action type distribution
plt.figure(figsize=(10, 6))
plt.bar(action_pd['action'], action_pd['count'])
plt.title('Activity Distribution by Action Type')
plt.xlabel('Action')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Action distribution plot created")

In [ ]:
# Device usage pie chart
plt.figure(figsize=(8, 8))
plt.pie(device_pd['usage_count'], labels=device_pd['device'], autopct='%1.1f%%', startangle=90)
plt.title('Device Usage Distribution')
plt.axis('equal')
plt.tight_layout()
plt.show()

print("Device distribution plot created")

## Part 8: Data Export

In [ ]:
# Export processed data
print("Exporting processed data...")

# Export user statistics
user_stats.write.mode("overwrite").parquet("output/user_statistics")
print("User statistics exported")

# Export action statistics
action_stats.write.mode("overwrite").csv("output/action_statistics", header=True)
print("Action statistics exported")

# Export engagement scores
engagement_df.write.mode("overwrite").parquet("output/engagement_scores")
print("Engagement scores exported")

## Part 9: Summary and Conclusions

In [ ]:
# Generate summary report
print("=" * 50)
print("ASSIGNMENT 2 SUMMARY REPORT")
print("=" * 50)

total_users = df_transformed.select("user_id").distinct().count()
total_activities = df_transformed.count()
date_range = df_transformed.agg(
    min("date").alias("start_date"),
    max("date").alias("end_date")
).collect()[0]

print(f"\nData Overview:")
print(f"Total Users: {total_users}")
print(f"Total Activities: {total_activities}")
print(f"Date Range: {date_range['start_date']} to {date_range['end_date']}")

print(f"\nKey Metrics:")
print(f"Average activities per user: {total_activities / total_users:.2f}")

top_action = action_stats.first()
print(f"Most common action: {top_action['action']} ({top_action['count']} times)")

top_device = device_stats.first()
print(f"Most used device: {top_device['device']} ({top_device['usage_count']} times)")

print("\nProcessing completed successfully!")
print("=" * 50)

In [ ]:
# Stop Spark session
spark.stop()
print("Spark session stopped")